In [8]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from MDP.STIRFutures.STIRFutureMDP import STIRFutureMDP
from MDP.MultiProductMDP import MultiProductMDP
from MDP.USTFutures.USTFuturesMDP import USTFuturesMDP




In [10]:
from SDRUtils.SDRDataBuilder import SDRDataBuilder
from SDRUtils.classification import classify_trade, classifications_to_dataframe
from SDRUtils.filters import filter_new_sofr_ois_trades 
from SDRUtils.package_detection import detect_fly_trades_df, detect_curve_trades_df, detect_spreadover_trades_df, _load_ust_reference_data
from SDRUtils.seasonality import add_event_classifications, analyze_seasonality_by_event, aggregate_flows_by_label

In [11]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

start = NY_tz.localize(datetime.datetime(2025, 1, 1, 7, 00))
end = NY_tz.localize(datetime.datetime(2025, 12, 26, 17, 00))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=filter_new_sofr_ois_trades,
)

MERGING REPORTS...: 100%|██████████| 247/247 [01:42<00:00,  2.42it/s]


In [13]:
from tqdm import tqdm

classifications = []
it = raw_df.iterrows()

it = tqdm(it, total=len(raw_df), desc="Classifying trades", unit="trade")

for idx, row in it:
	trade_id = int(row.get("Dissemination Identifier", idx))
	try:
		classification = classify_trade(row, trade_id)
		classifications.append(classification)
	except Exception as e:
		continue

classifications_df = classifications_to_dataframe(classifications)
with_pkg_df = detect_spreadover_trades_df(detect_curve_trades_df(detect_fly_trades_df(classifications_df)))
with_pkg_and_events_df = add_event_classifications(with_pkg_df, include_me=True)

Classifying trades: 100%|██████████| 862959/862959 [00:53<00:00, 16255.38trade/s]


In [30]:
with_pkg_df[with_pkg_df["package_type"] == "FLY"].tail(3).to_dict(orient="records")

[{'trade_id': 1558990689000001301,
  'execution_timestamp': Timestamp('2025-12-26 16:30:09+0000', tz='UTC'),
  'effective_date': Timestamp('2025-12-30 00:00:00'),
  'expiration_date': Timestamp('2034-12-30 00:00:00'),
  'product_type': 'OIS_SWAP',
  'tenor_years': 9.127777777777778,
  'tenor_label': '9Y',
  'is_forward': False,
  'forward_start_years': 0.011111111111111112,
  'forward_label': 'spot',
  'trade_label': 'spot 9Y',
  'notional': 13000000.0,
  'notional_currency': 'USD',
  'fixed_rate': 0.03706,
  'strike': None,
  'estimated_pv01': 11866.111111111113,
  'package_type': 'FLY',
  'package_id': 'FLY_4458',
  'package_legs': [1558990685000000901,
   1558990687000001101,
   1558990689000001301],
  'matched_ust_maturity': False,
  'ust_cusip': nan,
  'ust_oi': nan,
  'ust_issue_date': nan,
  'swap_maturity_date': datetime.date(2034, 12, 30)},
 {'trade_id': 1558990685000000901,
  'execution_timestamp': Timestamp('2025-12-26 16:30:09+0000', tz='UTC'),
  'effective_date': Timestamp